# 2-3: Text Analysis Introduction

In this notebook, we'll use Python to begin studying a literary text as data. The goal is not to replace close reading. The goal is to learn a few ways to ask computational questions about language, then decide what those results might mean. This is a common approach in DH. Python makes it possible!

We will use Bram Stoker's _Dracula_, saved locally as `dracula.txt` in the `data` folder.

## Fast Review

1. What is a dataframe?
2. What methods and other syntax can be used to subset rows and columns?
3. What is a Python library?
4. What is a string?
5. What is a list?


## Learning Objectives

By the end of this notebook, you should be able to:

1. Name a few common Python libraries for text analysis.
2. Use public-domain collections like Project Gutenberg and HathiTrust to find texts for DH research.
3. Load a `.txt` file into Python.
4. Preprocess a text by lowercasing, removing punctuation, tokenizing, and removing stop words.
5. Count words, unique words, and word frequencies.
6. Explain the basic difference between stemming and lemmatization.

## Text Analysis Libraries

Python has many libraries for working with text. Some are built into Python. Others need to be installed.

For today, we will try out the tools:

- `pathlib`: helps us work with file paths.
- `string`: gives us a ready-made list of common punctuation characters.
- `pandas`: our favorite library for interacting with tabular data.
- `matplotlib`: just to quick plots, like we've done before.
- `nltk`: provides common text analysis tools, including tokenization, frequency distributions, stop word lists, stemming, and lemmatization.

You may also hear about `spaCy` and `scikit-learn`. These are powerful libraries for more advanced natural language processing and machine learning. We explore them in later sessions. For now, we'll focus on more lightweight libraries.

Let's start by importing the necessary libraries and modules. As we're looking at them, can you tell which are libraries, modules, and aliases?

In [ ]:
from string import punctuation
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords

The `string` library gives us a variable called `punctuation`. It is just a string containing common punctuation marks. This is useful because––as you'll see below––punctuation often creates both opportunities and challenges for text analysis.

For example, consider `vampire`, `vampire.`, and `vampire,`. A human can read the difference and understand that `vampire` is the word and the `.` or `,` are technically not part of the word. But a computer using Python just understands them all as strings. So, a variable containing common `punctuation` is something we can use to help identify and manipulate punctuation in text.

In [ ]:
# what is the variable punctuation?
print(punctuation)
print(len(punctuation))

We've also got the library `nltk`. This stands for "natural language toolkit". It's a tried-and-true Python library for working with human language data.

Like other libraries, `nltk` has all sorts of useful methods and variables. You can learn more about `nltk` via its [documentation here](https://www.nltk.org/).

From `nltk`, we imported a variable `stopwords`. Let's see what this variable is:

In [ ]:
# run this twice, nltk's lazy loader won't tell you much the first time
help(stopwords)

Hmmm, it's a "List of words, one per line.  Blank lines are ignored.". So, a word list? What's in the list?

In [ ]:
# notice the nltk method .words()
stop_words = stopwords.words()
print(stop_words)
len(stop_words)

It's a really long list of words––the most common words, in fact. `nltk` uses the `.words()` method alongside this variable to call the list of words. The `.words()` method also takes arguments. 

For example, you can use it to subset the stopwords by English:

In [ ]:
stop_words = stopwords.words("english")
print(stop_words)
len(stop_words)

Why would we want a list of the most common words in English? Why design a Python library to contain this kind of list? Well, in many text analysis projects, the most common words are not always the most meaningful words for the questions we're asking. Words like “the,” “and,” “to,” and “of” appear so often that they can dominate our results, especially in word frequency counts. NLTK includes stopword lists so we don't have to create these lists from scratch every time. Instead, we can use a shared, reusable list as a starting point, then decide whether to keep, remove, or modify it depending on our research question.

## Finding Texts

But let's take one step back. DH projects often begin with the question: where can I get texts to study?

We'll discuss this in greater detail during our tutorials on webscraping and APIS, but for the sake of this tutorial, let's consider two useful collections of textual data: Project Gutenberg and HathiTrust.

Project Gutenberg provides public-domain ebooks, often as plain `.txt` files. Plain text is convenient because Python can read it without special software.

HathiTrust is a large digital library. The HathiTrust Research Center (HTRC) provides tools for computational research with texts, including extracted features and APIs. Some HathiTrust materials are limited by copyright, so always pay attention to access rules and terms of use.


### HathiTrust

HathiTrust is especially useful when you want to work with many texts, not just one downloaded file.

For a first text analysis project, you might use HathiTrust to:

- search for books related to an author, genre, period, or topic;
- build a collection of texts;
- learn what can and cannot be downloaded because of copyright.

I mention it here because you could download a .txt file from HathiTrust and work through the steps below with only minor changes. It's a great resource for data that you may want to utilize for your project. 

Learn more about [HathiTrust here](https://www.hathitrust.org/).

For today, we will keep working with one local file so we can focus on Python basics.

### Project Gutenberg

Project Gutenberg is another large resource for texts and textual data. Everything in Project Gutenberg has been hand-keyed (transcribed by humans), meaning it is generally more accurate than Optical Character Recognition (OCR) texts. To download texts from Project Gutenberg, simply [visit the site here](https://www.gutenberg.org/), search for texts, and download them as `.txt` files.

The file for this lesson has already been downloaded for you: `dracula.txt`.

If you wanted to recreate some of these steps with a different text from Project Gutenberg, you could do so fairly easily.

Because different computers have different absolute file paths, we'll use a relative path. If this notebook is in the `week_2` folder and the text is in the `data` folder, then the path is:

`../data/dracula.txt`

Do you remember what that pattern means? Go up one folder, then go into the `data` folder.

In [ ]:
import os

os.getcwd()

# open function takes path and encoding arguments
dracula_file = open("../data/dracula.txt", encoding="utf-8-sig")

Now we've got the path to `dracula.txt` as the object `dracula_file`. But keep in mind: `open()` just creates the path to the file. It doesn't actually contain the text.

To get the text, we'll use the standard library method `.read()`:

In [ ]:
# text of dracula to dracula_raw
dracula_raw = dracula_file.read()

# notice we can use .close() to reverse our .open() pathway to the file
dracula_file.close()

# what's in dracula_raw?
print(type(dracula_raw))
print(len(dracula_raw))
print(dracula_raw[:1000])


The variable `dracula_raw` contains the whole file as one long string!

Notice that the beginning includes Project Gutenberg information. This is useful metadata, but it's not part of the novel itself. Many downloaded texts include headers, footers, tables of contents, notes, or other extra material. Before analysis, we often need to decide what to keep.

But what if we wanted to just get the text of Dracula, not all this Gutenberg stuff?

Frankly, there's a lot of ways to do this, but one of the easiest is to review the text and create __textual markers__ for Python to identify. These should be unique, long, not something that repeats frequently in the text. Thankfully, Gutenberg labels its metadata pretty clearly. If you scroll to the end of dracula.txt, you'll also notice the text ends with the usual "THE END" in all caps.

To remove the Gutenberg boilerplate, we'll make a `start_marker` and `end_marker`:

In [ ]:
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK DRACULA ***"
end_marker = "THE END"

Next we'll use the standard library method `.find()` to index these markers, like this:

In [ ]:
start_index = dracula_raw.find(start_marker)
end_index = dracula_raw.find(end_marker)

And now, let's see their indices:

In [ ]:
print(start_index)
print(end_index)

The `.find()` method returned the character index where a string first appears. We can use those index numbers to slice away the Project Gutenberg header and footer.

You'll find that a lot of text analysis in Python involves identifying word and character indices. It's a funny thing, but you'll start to think of texts as long, indexable strings!

But let's slice away. We'll take the `start_index` (the 740th character) and add to it the length of the `start_marker` (the Gutenberg "START OF THE ... EBOOK" text), and then run it all the way to the `end_index`. To do that, we can use the `.strip()` method.

In [ ]:
# pull just the text of Dracula

dracula_body = dracula_raw[start_index + len(start_marker):end_index].strip()

print(len(dracula_body))
print(dracula_body[:1000])

It worked! But there's still some front matter before the story begins. If you look at dracula.txt in a text editor, you'll see the story actually starts with Jonathan Harker's first diary entry. Let's see if we can strip some more so our data starts there. This keeps the example simple and gives us mostly narrative text.

In this case, we want all the text from the first line "3 May. Bistritz" to the end of the document since we already removed the ending boilerplate with `end_index`. 

In [ ]:
# use .find() to mark the start of the text
novel_start = dracula_body.find("3 May. Bistritz")

# strip from the novel_start index to the end of the document
dracula = dracula_body[novel_start:].strip()

Did that work? Let's check the length of dracula:

In [ ]:
len(dracula)

That's over 840,000 characters. Sounds about right. And how can we check the starting location text?

In [ ]:
print(dracula[:1000])

And the end location?

In [ ]:
print(dracula[-1000:])

## Text Preprocessing

What we've already done is a small example of what's called __text preprocessing__. Text preprocessing means preparing text for analysis. We've already removed boilerplate from the data source (Gutenberg), but there are lots of other potential preprocessing steps we'll need to do.

There is no single correct way to preprocess a text. Your choices depend on your research question. If punctuation matters to your question, you might keep it. If capitalization matters, you might keep it. In this workflow, let's imagine we want to do some word frequency analysis––that is, we want to count instances of words as a way to understand the text.

Let's say we wanted to count the instances of the word "blood". It's Dracula, after all! While conceptually, counting words is straightforward, we need to keep in mind that the computer doesn't read words the same way we do. It will count "Blood", "blood", "blood.", and "blood," as completely different instances of the word. It also won't be immediately clear to the computer how to parse examples like "bloodlust" or "bloody". Remember: you must think in strings. How can we help the computer parse the text as one long string?

Some common preprocessing steps include:

- lowercasing the text
- removing punctuation
- tokenizing (setting word boundaries)

Again, your approach to preprocessing will depend on your research questions. Be sure to keep them in mind as you move through preprocessing workflows.


### Counting Characters, Words, and Lines

But before more preprocessing, we should ask simple counting questions. These methods come from Python strings and lists, not from a special text analysis library. They allow us to get a sense of the shape our "unprocessed" text.

We can count the characters easily with 'len()'. We can get a rough sense of the number of words with .`split()` (by default, separates words whenever there's a whitespace), and using the "\n" newline character, we can count the lines in the text:

In [ ]:
print("Characters:", len(dracula))
print("Words from simple split:", len(dracula.split()))
# notice the "\n" character
print("Lines:", len(dracula.split("\n")))

Counting sentences is trickier than counting words. A first attempt might count periods or split on periods, but abbreviations and initials also use periods.

This example is intentionally imperfect. It shows why text analysis often requires interpretation, not just code.


In [ ]:
# What are each of these lines counting, precisely?
print("Periods:", dracula.count("."))
print("Pieces of text after splitting on periods:", len(dracula.split(".")))

### A Small Sample

Before changing the whole text with preprocessing steps, it's good practice to test your preprocessing code on a sample. This lets us see exactly what each step does and make sure we're changing or processing what we intend to. So, let's get a sample of the text:

In [ ]:
sample = dracula[:1200]

print(sample)

### Tokenization

Tokenization means splitting text into smaller pieces called __tokens__. Most often, we tokenize text into words, but you don't have to think of tokens as strictly words. They are just predetermined parts of a text. Sentences can be tokens. Characters can be tokens. Paragraphs, pages. A token is just a unit of the text.

#### Simple Tokenization with `.split()`

The simplest Python tokenizer is `.split()`, which we've already used. By default, `.split()` separates a string wherever it finds whitespace:

In [ ]:
raw_tokens = sample.split()

print(raw_tokens[:80])
print("Number of tokens in sample:", len(raw_tokens))


This method of tokenization for words is okay, but notice how it treats "P. M." as two tokens? And "Bistritz._--Left" as one token? In a small sample, these may not equate to a major counting error, but across long texts, they can add up to major inaccuracies.

Let's tokenize the whole novel with the same method and see the results:

In [ ]:
dracula_raw_tokens = dracula.split()

print(dracula_raw_tokens[:40])
print("Total raw tokens:", len(dracula_raw_tokens))

#### Tokenization with NLTK

The `.split()` method is useful because it is simple and built into Python. But it only knows how to split strings at whitespace or via some marker that we select for it. It doesn't really understand punctuation, sentence boundaries, or common patterns in written language.

`nltk` gives us tokenizers that are designed specifically for text analysis. The method we'll use here is `nltk.word_tokenize()`. It still isn't perfect, but it usually handles punctuation better than `.split()` because it can separate many punctuation marks into their own tokens.

Other libraries can tokenize text, too, including `spaCy`, `TextBlob`, and `scikit-learn`. For now, we'll compare `.split()` and `nltk.word_tokenize()` using the same `sample` variable from above. That way, we can see what changes and what stays the same.

In [ ]:
# If this is your first time using nltk tokenization on this computer,
# you may need to download the tokenizer data.
nltk.download("punkt")
nltk.download("punkt_tab")

# Tokenize the same sample with nltk word_tokenize() method
nltk_sample_tokens = nltk.word_tokenize(sample)

print(nltk_sample_tokens[:80])
print("Number of tokens in sample with .split():", len(raw_tokens))
print("Number of tokens in sample with nltk:", len(nltk_sample_tokens))

Compare this output to the `.split()` output above. Both methods begin with the same text, but they do not create exactly the same tokens.

With `.split()`, a token like `May.` stays together because there is no space between `May` and the period. With `nltk.word_tokenize()`, `May` and `.` are separated. Likewise, commas and semicolons often become their own tokens.

This means nltk may produce more tokens than `.split()`. That's not automatically better or worse. It depends on what you want to count. The important thing is to know which tokenization method you used and why.

In [ ]:
# Tokenize the full Dracula text with nltk
dracula_nltk_tokens = nltk.word_tokenize(dracula)

print(dracula_nltk_tokens[:40])
print("Total raw tokens with .split():", len(dracula_raw_tokens))
print("Total raw tokens with nltk:", len(dracula_nltk_tokens))

### Counting Unique Words

There are built-in Python functions that let us count unique words, too. They rely on another data structure we haven't yet covered: `sets`. A `set` is like a list, but it stores only unique values. If we convert a list of tokens into a set, repeated words collapse into one copy.

This is useful, but look carefully: before cleaning, punctuation and capitalization affect the count.

In [ ]:
# convert raw tokens to a set of unique words
raw_unique_words = set(dracula_raw_tokens)

print("Raw unique words:", len(raw_unique_words))
print(list(raw_unique_words)[:50])

Notice how it's counting strange things like unique words? It's even counting uppercase and lowercase differences as unique tokens. What happens if we lowercase the text before converting it to a set?

In [ ]:
# remember .lower() method of working with strings?
lowercase_unique_words = set(dracula.lower().split())

print("Raw unique words:", len(raw_unique_words))
print("Lowercase unique words:", len(lowercase_unique_words))

Why did the number change? Because examples like `The` and `the` are different strings to Python. Lowercasing makes them match.


#### Counting Unique Words with nltk

You can also count unique words with nltk's `FreqDist` method. "FreqDist" means frequency distribution. It stores each unique token once, along with the number of times it appears.

Because `FreqDist` stores each unique token once, we can also use `len()` on a `FreqDist` object to count unique words. This makes it comparable to the `set()` examples above. Let's use the same token lists so we can compare the results directly.


In [ ]:
# Create a frequency distribution from the same raw tokens
raw_freq_dist = nltk.FreqDist(dracula_raw_tokens)

# Create a frequency distribution from the same lowercased tokens
lowercase_freq_dist = nltk.FreqDist(dracula.lower().split())

print("Raw unique words with set():", len(raw_unique_words))
print("Raw unique words with FreqDist:", len(raw_freq_dist))
print("Lowercase unique words with set():", len(lowercase_unique_words))
print("Lowercase unique words with FreqDist:", len(lowercase_freq_dist))

The results are the same, so what's the difference? Well, `FreqDist` also keeps track of __word frequencies__. Let's look at the objects we've created to understand the difference:

In [ ]:
print(type(raw_unique_words))
print(raw_unique_words)

In [ ]:
print(type(raw_freq_dist))
raw_freq_dist

### Removing Punctuation

Another common preprocessing step is removing punctuation. Sometimes, this can make text analyses more straightforward.

There's a simple way to do this: just use the `.replace()` method. If you wanted to remove all periods from our sample, for example, you could do it like this:

In [ ]:
# notice how .replace() takes string then other string (empty string here)
sample_no_periods = sample.replace(".", "")

print(sample_no_periods[:500])

The text still contains commas, colons, quotation marks, parentheses, underscores, and other punctuation. We could keep running replace and feeding it different punctuation, one by one, but here's where some of those imports become useful.

Remember `punctuation`? The variable we imported from `string`?

Rather than using replace for punctuation one by one, let's loop through the `punctuation` list and remove accordingly. Let's also start working toward a fully "cleaned" version of our text. We'll call it `dracula_clean`.

In [ ]:
# quick lowercase
dracula_clean = dracula.lower()

# a simple for loop to remove punctuation
for character in punctuation:
    dracula_clean = dracula_clean.replace(character, "")

print(dracula_clean[:1000])

This simple method is helpful, but it is not perfect. The `punctuation` variable mainly contains common ASCII punctuation. Some texts include curly quotation marks, long dashes, accented characters, or other symbols. Real-world text cleaning often takes multiple passes.


### Tokenization After Cleaning

Now that we've removed punctuation and lowercased our text, we can tokenize a little more easily. These tokens should be easier to count because most punctuation and capitalization differences have been removed.

Let's start with the `.split()` method first. What does it yield?

In [ ]:
dracula_tokens = dracula_clean.split()

print(dracula_tokens[:40])
print("Clean tokens:", len(dracula_tokens))
print("Clean unique words:", len(set(dracula_tokens)))


How about if we do it with nltk? How much of a difference does it make?

In [ ]:
# Tokenize the full Dracula text with nltk
dracula_nltk_tokens_clean = nltk.word_tokenize(dracula_clean)

print(dracula_nltk_tokens_clean[:40])
print("Clean tokens with nltk:", len(dracula_nltk_tokens_clean))

# Count unique words with NLTK FreqDist
dracula_nltk_freq_dist_clean = nltk.FreqDist(dracula_nltk_tokens_clean)
print("Clean unique words with nltk FreqDist:", len(dracula_nltk_freq_dist_clean))


### Counting Words

Once the text is tokenized, counting words is just list work. We can count total words with `len()` and unique words with `set()`.


In [ ]:
total_words = len(dracula_tokens)
unique_words = len(set(dracula_tokens))

print("Total words:", total_words)
print("Unique words:", unique_words)
print("Type-token ratio:", unique_words / total_words)


The type-token ratio is one simple measure of lexical variety: unique words divided by total words. It can be interesting, but it is also sensitive to text length and preprocessing choices.


### Counting Word Frequencies

Now that we have introduced `FreqDist`, we can use it for word frequency counting too. This is the same object we used to count unique words. The difference is that now we want to ask which words appear most often.

The `.most_common()` method shows the most frequent tokens in a `FreqDist` object. Since we already created `dracula_nltk_freq_dist_clean` from the cleaned Dracula tokens, we can use that counted data directly.


In [ ]:
# Show the 20 most common tokens in the cleaned dracula text
dracula_nltk_freq_dist_clean.most_common(20)


The most common words are mostly short function words like `the`, `and`, and `to`. These are often called __stop words__. Stop words can be meaningful, but if we want to see topic-heavy words, they often get in the way.


### Stop Word Removal

Can you think of what might be considered stop words? What words do you think are the most common in the English language?

Now imagine trying to type out those words every time you wanted to remove them from textual data... Time-consuming! That's why nltk includes a standard English stop word list. Do you remember? We already imported it into this session as `stop_words`.

In [ ]:
print(len(stop_words))
print(stop_words)

It's 179 words long. Let's use it to remove stop words from our FreqDist object `dracula_nltk_freq_dist_clean`. We can do that by looping through the object and removing words as they appear in both the object and the `stop_words` list, like this:

In [ ]:
# create a shell object (a new FreqDist object) to hold our no-stopword FreqDist
dracula_no_stops_freq_dist = nltk.FreqDist()

# loop passes the word and its count
for word, count in dracula_nltk_freq_dist_clean.items():
    # if it's not in stop_words, add it to no-stops
    if word not in stop_words:
        dracula_no_stops_freq_dist[word] = count

# let's see what we got!
dracula_no_stops_freq_dist.most_common(20)

This already seems more insightful, yes? Words like "said" or "us" or character names like "van" and "helsing" tell you something about the text.

What information can you glean from this top-20 list of words?

We can also see how much of the text is removed when we remove stop words. Let's do another for-loop to calculate it:

In [ ]:
# shell to hold words not removed (i.e., not stop words)
dracula_no_stops = []

# for every word in dracula_tokens
for word in dracula_tokens:
    # check if word is not in stop_words
    if word not in stop_words:
        # if not in stop_words, add it to our shell
        dracula_no_stops.append(word)

print(dracula_no_stops[:80])
print("Tokens before stop word removal:", len(dracula_tokens))
print("Tokens after stop word removal:", len(dracula_no_stops))


That's like... almost half the text?! Turns out, most of our language is the little stuff: articles, prepositions, etc.

Let's turn back to our list of most frequent, non-stop-word words:

In [ ]:
dracula_no_stops_freq_dist.most_common(20)


Reading this frequency list is fine, but it's often easier to read it as a dataframe. This also connects our text analysis work back to pandas.

Here's how we can turn it into a dataframe:

In [ ]:
frequency_df = pd.DataFrame(
    dracula_no_stops_freq_dist.most_common(25),
    columns=["word", "count"]
)

frequency_df


That's easy enough! And now, we can visualize it with matplotlib, like this:

In [ ]:
frequency_df.plot(kind="bar", x="word", y="count", legend=False)

plt.title("Most Common Words in Dracula")
plt.xlabel("Word")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Interpreting Frequencies

Word frequencies can quickly show patterns, but they do not explain themselves.

For example, if a character name appears often, that might tell us something about narration, plot, or point of view. But it might also reflect chapter structure, diary entries, or preprocessing choices. Use frequency counts as a starting point for better questions.

What questions do you find yourself considering when we look at the frequencies of these words?


In [ ]:
search_words = ["dracula", "lucy", "mina", "harker", "vampire", "blood"]

for word in search_words:
    print(word, dracula_no_stops_freq_dist[word])


### Stemming and Lemmatization

Stemming and lemmatization both try to reduce related words to a shared form. This can be useful when we want Python to count related forms of a word together instead of treating each form as completely separate. For example, dream, dreamed, and dreaming are different strings, but they are clearly related words.

Stemming usually cuts words down using rules. It is fast, but the result may not be a real word. For example, a stemmer might reduce "studies" to something like "studi". That may look strange to us as readers, but it can still be useful for counting and comparison.

Lemmatization tries to return a dictionary form, or lemma. It is often more readable, but it needs more linguistic information. For example, "studies" might become "study", and "vampires" might become "vampire". Lemmatization can also depend on part of speech: "dreaming" as a verb should become "dream", but without that information, Python may not always make the change we expect.

Here's how you could preprocess some text by stemming via nltk:


In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

words_to_stem = [
    "vampire", "vampires", "dream", "dreamed", "dreaming",
    "studies", "studied", "flies", "flying"
]

for word in words_to_stem:
    print(word, "->", stemmer.stem(word))


Let's also stem the whole list of non-stopword tokens, then count again. Notice that stems may be less readable than full words.

In [ ]:
stemmed_tokens = []

for word in dracula_no_stops:
    stemmed_tokens.append(stemmer.stem(word))

stemmed_counts = nltk.FreqDist(stemmed_tokens)
print(stemmed_counts.most_common(20))
print(dracula_no_stops_freq_dist.most_common(20))


Lemmatization in nltk uses a resource called WordNet. If the WordNet data is not installed, the cell below will try to download it.

In [ ]:
from nltk.stem import WordNetLemmatizer

# Download the data needed for lemmatization
nltk.download("wordnet")
nltk.download("omw-1.4")


In [ ]:
lemmatizer = WordNetLemmatizer()

words_to_lemmatize = [
    "vampires", "dreamed", "dreaming",
    "studies", "studied", "flies", "flying"
]

for word in words_to_lemmatize:
    noun_lemma = lemmatizer.lemmatize(word)
    verb_lemma = lemmatizer.lemmatize(word, pos="v")
    print(word, "-> noun:", noun_lemma, "verb:", verb_lemma)